In [0]:
%pip install hyperopt

In [0]:
import numpy as np
import pandas as pd
import sklearn.datasets
import sklearn.metrics
import sklearn.model_selection
import sklearn.ensemble


from hyperopt import fmin, tpe, hp, SparkTrials, Trials, STATUS_OK
from hyperopt.pyll import scope

In [0]:
import mlflow
mlflow.set_registry_uri("databricks")

In [0]:
CATALOG_NAME = "my_catalog"
SCHEMA_NAME = "my_schema"

In [0]:
white_wine = spark.read.csv(
    "/databricks-datasets/wine-quality/winequality-white.csv",
    sep=';',
    header=True,
    inferSchema=True
)
white_wine.limit(5).display()

In [0]:
red_wine = spark.read.csv(
    "/databricks-datasets/wine-quality/winequality-red.csv",
    sep=';',
    header=True,
    inferSchema=True
)
red_wine.limit(5).display()

In [0]:
# Remove the spaces from column names
for c in white_wine.columns:
    white_wine = white_wine.withColumnRenamed(c, c.replace(' ', '_'))
for c in red_wine.columns:
    red_wine = red_wine.withColumnRenamed(c, c.replace(' ', '_'))

In [0]:
print(white_wine.columns)
print(red_wine.columns)

In [0]:
# define table names
white_wine_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.white_wine"
red_wine_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.red_wine"

In [0]:
#Write to tables in Unity Catalog
spark.sql(f"DROP TABLE IF EXISTS {white_wine_table}")
spark.sql(f"DROP TABLE IF EXISTS {red_wine_table}")
white_wine.write.saveAsTable(white_wine_table)
red_wine.write.saveAsTable(red_wine_table)

In [0]:
white_wine = spark.table(white_wine_table).toPandas()
red_wine = spark.table(red_wine_table).toPandas()

In [0]:
display(white_wine.head(5))
display(red_wine.head(5))

In [0]:
# Add Boolean fields for red and white wine
white_wine['is_red'] = 0
red_wine['is_red'] = 1

# Combine the two DataFrames
df_wine = pd.concat([white_wine, red_wine], ignore_index=True)
display(df_wine.head(5))
display(df_wine.tail(5))

In [0]:
# Define classification labels based on the wine quality
data_labels = df_wine['quality'].astype('int')>=7
data_labels

In [0]:
# Dropping quality column to create feature dataset
df_wine = df_wine.drop(['quality'], axis=1)
display(df_wine.head())

In [0]:
# Split 80/20 train-test
X_train, X_test, y_train, y_test = sklearn.model_selection.train_test_split(
    df_wine,
    data_labels,
    test_size = 0.2,
    random_state = 1
)

In [0]:
mlflow.autolog()

In [0]:
with mlflow.start_run(run_name='gradient_boost') as run:
    model = sklearn.ensemble.GradientBoostingClassifier(random_state=0)

    # Models, parameters and training metrics are tracked automatically
    model.fit(X_train, y_train)

    predicted_probs = model.predict_proba(X_test)
    roc_auc = sklearn.metrics.roc_auc_score(y_test, predicted_probs[:,1])

    # The AUC score on test data is not automatically logged, so log it manually
    mlflow.log_metric("test_auc", roc_auc)
    print("Test AUC of: {}".format(roc_auc))

In [0]:
with mlflow.start_run(run_name='gradient_boost') as run:
    model = sklearn.ensemble.GradientBoostingClassifier(
        random_state=0,
        n_estimators = 200
    )

    # Models, parameters and training metrics are tracked automatically
    model.fit(X_train, y_train)

    predicted_probs = model.predict_proba(X_test)
    roc_auc = sklearn.metrics.roc_auc_score(y_test, predicted_probs[:,1])

    # The AUC score on test data is not automatically logged, so log it manually
    mlflow.log_metric("test_auc", roc_auc)
    print("Test AUC of: {}".format(roc_auc))

In [0]:
run_id = run.info.run_id
# Load the model back from the run
model_uri = f"runs:/{run_id}/model"
model_loaded = mlflow.pyfunc.load_model(model_uri)


predictions_loaded = model_loaded.predict(X_test)
predictions_original = model.predict(X_test)

# The loaded model should match the original
assert np.array_equal(predictions_loaded, predictions_original)

In [0]:
search_space = {
    'n_estimators': scope.int(hp.quniform('n_estimators', 20, 1000, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'max_depth': scope.int(hp.quniform('max_depth', 2, 5, 1))
}

In [0]:
def train_model(params):
    # Enable autologging on each worker
    mlflow.autolog()
    with mlflow.start_run(nested=True):
        model_hp = sklearn.ensemble.GradientBoostingClassifier(
            random_state=0,
            **params
        )
        model_hp.fit(X_train, y_train)
        predicted_probs = model_hp.predict_proba(X_test)
        roc_auc = sklearn.metrics.roc_auc_score(y_test, predicted_probs[:,1])
        mlflow.log_metric("test_auc", roc_auc)
        return {'status': STATUS_OK, 'loss': -1*roc_auc}

In [0]:
spark_trials = Trials()

In [0]:
with mlflow.start_run(run_name='gb_hyperopt') as run:
    best_params = fmin(
        fn=train_model,
        space=search_space,
        algo=tpe.suggest,
        max_evals=2,
        trials = spark_trials
    )

In [0]:
best_run = mlflow.search_runs(
    order_by = ['metrics.test_auc DESC', 'start_time DESC'],
    max_results = 10,
).iloc[0]

print('Best Run')
print('AUC: {}'.format(best_run['metrics.test_auc']))
print('Num Estimators: {}'.format(best_run['params.n_estimators']))
print('Max Depth: {}'.format(best_run['params.max_depth']))
print('Learning Rate: {}'.format(best_run['params.learning_rate']))


In [0]:
best_model_pyfunc = mlflow.pyfunc.load_model(
    'runs:/{run_id}/model'.format(
        run_id = best_run['run_id']
    )
)

best_model_predictions = X_test
best_model_predictions['predicted'] = best_model_pyfunc.predict(X_test)

In [0]:
predictions_table = f"{CATALOG_NAME}.{SCHEMA_NAME}.predictions"
spark.sql(f"DROP TABLE IF EXISTS {predictions_table}")

results = spark.createDataFrame(best_model_predictions).write.saveAsTable(predictions_table)

In [0]:
model_uri = 'runs:/{run_id}/model'.format(run_id = best_run['run_id'])
mlflow.set_registry_uri("databricks-uc")
# Register the model
model_registered = mlflow.register_model(model_uri, f"{CATALOG_NAME}.{SCHEMA_NAME}.wine_model")